In [ ]:
"""
Variable Rate Application (VRA) Zoning  +  SOC Spatial Map
===========================================================
Satellite : Sentinel-2 L2A  (10-m native resolution)
Sources   : AWS Element84 Earth-Search  +  Microsoft Planetary Computer
No GEE    : Pure pystac-client + rasterio COG reads

What this script produces
--------------------------
1.  SOC spatial map      — base64 PNG  +  per-pixel text stats
2.  VRA zone map (N/P/K) — base64 PNG  +  per-zone application-rate table
3.  Combined 4-panel map — base64 PNG  (SOC + N-zones + P-zones + K-zones)
4.  Full JSON-serialisable result dict

VRA logic
----------
• Field is divided into management zones based on satellite-derived soil
  nutrient content (Low / Medium / High for each nutrient).
• Application rate is INVERSE of soil content:
    zone with LOW  soil N  → HIGH  N fertiliser dose
    zone with MED  soil N  → MED   N fertiliser dose
    zone with HIGH soil N  → LOW   N fertiliser dose
• Per-zone dose is calculated relative to the crop's nutrient demand
  (from ICAR/FAO uptake tables).

Spectral indices used (all from peer-reviewed literature)
----------------------------------------------------------
NDVI   (Rouse 1973)      : (B08-B04)/(B08+B04)          vegetation / SOC proxy
BSI    (Rikimaru 2002)   : ((B11+B04)-(B08+B02))/((B11+B04)+(B08+B02))  bare soil
SAVI   (Huete 1988)      : ((B08-B04)/(B08+B04+0.5))*1.5                P proxy
SWIR1  direct B11        : shortwave-infrared reflectance                moisture/clay
CLAY   (Drury 1987)      : B11/B12                                       clay minerals
NDWI   (Gao 1996)        : (B08-B12)/(B08+B12)                          water content

Install
-------
pip install pystac-client rasterio pyproj shapely scipy numpy matplotlib affine

Usage
-----
python vra_zoning.py
"""

# ──────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, io, base64, math, json, warnings
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import nullcontext
from datetime import datetime, timedelta

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.windows import transform as win_transform
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.warp import reproject
from affine import Affine

from pystac_client import Client
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
from pyproj import Transformer
from scipy.ndimage import gaussian_filter

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# GDAL / AWS env  (public COGs — no credentials needed)
# ──────────────────────────────────────────────────────────────────────────────
_ENV = {
    "CPL_VSIL_CURL_USE_HEAD":           "FALSE",
    "GDAL_DISABLE_READDIR_ON_OPEN":     "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.tiff,.jp2,.TIF,.TIFF,.JP2",
    "AWS_NO_SIGN_REQUEST":              "YES",
    "AWS_REQUEST_PAYER":                "requester",
    "GDAL_HTTP_MULTIRANGE":             "YES",
    "GDAL_CACHEMAX":                    "512",
    "CPL_VSIL_CURL_CHUNK_SIZE":         "32768",
}
for k, v in _ENV.items():
    os.environ.setdefault(k, v)

# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
EARTH_SEARCH_URL = "https://earth-search.aws.element84.com/v1"
PLANETARY_URL    = "https://planetarycomputer.microsoft.com/api/stac/v1"

MAX_CLOUD      = 20          # strict pass
FALLBACK_CLOUD = 100         # relaxed pass
TRY_N          = 8           # STAC items per query
THREADS        = min(8, (os.cpu_count() or 4))
SMOOTH_SIGMA   = 2.0         # gaussian smoothing (pixels) for index maps
USE_SHADOW     = True        # mask SCL cloud-shadow class

MIN_PX_LONG = 400
MAX_PX_LONG = 1200
MIN_RES_M   = 0.5
MAX_RES_M   = 40.0

DPI = 150   # output image DPI

# ──────────────────────────────────────────────────────────────────────────────
# CROP NUTRIENT DEMAND TABLE  (kg nutrient / tonne of yield, ICAR/FAO)
# ──────────────────────────────────────────────────────────────────────────────
CROP_DEMAND: dict[str, dict] = {
    "wheat":        {"N": 12.5, "P": 5.0, "K": 10.5},
    "rice":         {"N": 15.0, "P": 5.5, "K": 12.0},
    "maize":        {"N": 18.0, "P": 6.0, "K": 14.0},
    "soybean":      {"N":  8.0, "P": 8.0, "K": 10.0},
    "sugarcane":    {"N": 22.0, "P": 9.0, "K": 20.0},
    "cotton":       {"N": 18.0, "P": 7.0, "K": 14.0},
    "onion":        {"N": 14.0, "P": 7.0, "K": 14.0},
    "potato":       {"N": 20.0, "P":10.0, "K": 22.0},
    "tomato":       {"N": 16.0, "P": 8.0, "K": 18.0},
    "banana":       {"N": 22.0, "P":10.0, "K": 28.0},
    "groundnut":    {"N":  8.5, "P": 7.0, "K":  8.5},
    "jowar":        {"N": 17.0, "P": 5.5, "K": 13.0},
    "bajra":        {"N": 14.5, "P": 4.5, "K": 11.0},
    "chili":        {"N": 18.0, "P": 8.0, "K": 15.0},
    "turmeric":     {"N": 22.0, "P":10.0, "K": 24.0},
    "ginger":       {"N": 22.0, "P":10.0, "K": 24.0},
    "mustard":      {"N": 14.0, "P": 7.0, "K": 11.0},
    "lentil":       {"N":  7.5, "P": 7.5, "K":  9.0},
    "gram":         {"N":  7.0, "P": 7.0, "K":  9.0},
    "default":      {"N": 14.0, "P": 7.0, "K": 12.0},
}

# Target full-season fertiliser doses (kg/ha) — ICAR Kharif/Rabi norms
# Used as MAXIMUM dose applied in low-nutrient zones
FERTILISER_MAX_DOSE = {
    "N": 150,   # kg/ha  (as Urea)
    "P": 60,    # kg/ha  (as DAP / SSP)
    "K": 100,   # kg/ha  (as MOP)
}

# Fertiliser products and nutrient content (%)
FERTILISER_PRODUCTS = {
    "N": {"name": "Urea",           "nutrient_pct": 46.0, "unit": "kg/ha"},
    "P": {"name": "DAP (18-46-0)",  "nutrient_pct": 46.0, "unit": "kg/ha"},
    "K": {"name": "MOP (0-0-60)",   "nutrient_pct": 60.0, "unit": "kg/ha"},
}

# Zone dose fractions (relative to MAX_DOSE)
ZONE_DOSE_FRACTION = {
    "High":   0.30,   # soil is rich  → apply only 30 % of max dose
    "Medium": 0.65,   # moderate soil → apply 65 %
    "Low":    1.00,   # poor soil     → apply 100 % (full dose)
}

# ──────────────────────────────────────────────────────────────────────────────
# COLOUR MAPS
# ──────────────────────────────────────────────────────────────────────────────
# SOC: warm brown (very low) → tan → yellow-green → rich green (very high)
_SOC_COLORS = ["#3d1c00", "#7a3b00", "#c17f00", "#e8c840", "#9ecb3c",
               "#4caf50", "#1b5e20"]
CMAP_SOC = LinearSegmentedColormap.from_list("soc", _SOC_COLORS, N=256)

# N zones:  deep red (Low soil = High dose needed) → amber → forest green
ZONE_COLORS = {
    "Low":    "#d32f2f",   # red    — soil deficient, high dose zone
    "Medium": "#f9a825",   # amber  — moderate
    "High":   "#388e3c",   # green  — soil rich, low dose needed
}
ZONE_INT = {"Low": 1, "Medium": 2, "High": 3}

# ──────────────────────────────────────────────────────────────────────────────
# STAC / COG HELPERS  (identical to proven NDVI + soil pipelines)
# ──────────────────────────────────────────────────────────────────────────────
def _s3_to_https(href: str) -> str:
    if href.startswith("s3://sentinel-cogs/"):
        return href.replace("s3://sentinel-cogs/",
                            "https://sentinel-cogs.s3.amazonaws.com/")
    if href.startswith("s3://"):
        bucket, key = href[5:].split("/", 1) if "/" in href[5:] else (href[5:], "")
        return f"https://{bucket}.s3.amazonaws.com/{key}"
    return href


def _prefer_https(asset) -> str | None:
    if asset is None:
        return None
    href = getattr(asset, "href", "") or ""
    alt  = getattr(asset, "extra_fields", {}).get("alternate", {})
    for k in ("https", "http", "self"):
        v = alt.get(k)
        if isinstance(v, dict):
            url = v.get("href", "")
            if url.startswith("http"):
                return url
        elif isinstance(v, str) and v.startswith("http"):
            return v
    if href.startswith("http"):
        return href
    return _s3_to_https(href) if href else None


def _pick_url(assets: dict, *keys) -> str | None:
    for k in keys:
        a = assets.get(k)
        if a:
            url = _prefer_https(a)
            if url:
                return url
    return None


def _aoi_scene(aoi_ll: dict, crs_str: str):
    t = Transformer.from_crs("EPSG:4326", crs_str, always_xy=True)
    return shp_transform(lambda x, y, z=None: t.transform(x, y),
                         shape(aoi_ll))


# ──────────────────────────────────────────────────────────────────────────────
# SCENE SEARCH  (AWS → Planetary Computer fallback)
# ──────────────────────────────────────────────────────────────────────────────
def _stac_search(catalog: str, aoi: dict,
                 start: str, end: str, max_cloud: int, n: int) -> list:
    try:
        cat = Client.open(catalog)
        return list(cat.search(
            collections=["sentinel-2-l2a"],
            intersects=aoi,
            datetime=f"{start}/{end}",
            sortby=[{"field": "properties.datetime", "direction": "desc"}],
            query={"eo:cloud_cover": {"lt": max_cloud}},
            limit=n,
        ).items())
    except Exception as exc:
        print(f"  [STAC] {catalog} → {exc}")
        return []


def find_best_scene(aoi: dict, start: str, end: str) -> tuple:
    """
    Returns (item, capture_date, cloud_pct).
    Strategy: strict cloud → relaxed cloud → extended date window.
    """
    fmt = "%Y-%m-%d"
    windows = [
        (start, end, MAX_CLOUD),
        (start, end, FALLBACK_CLOUD),
        ((datetime.strptime(start, fmt) - timedelta(days=30)).strftime(fmt),
         (datetime.strptime(end,   fmt) + timedelta(days=30)).strftime(fmt),
         FALLBACK_CLOUD),
    ]
    for s, e, mc in windows:
        for cat_url in [EARTH_SEARCH_URL, PLANETARY_URL]:
            items = _stac_search(cat_url, aoi, s, e, mc, TRY_N)
            if items:
                it    = items[0]
                cap   = (it.properties.get("datetime") or "")[:10]
                cloud = it.properties.get("eo:cloud_cover", "NA")
                print(f"  [Scene] {cap}  cloud={cloud}%  src={cat_url.split('/')[2]}")
                return it, cap, cloud
    return None, "NA", "NA"


# ──────────────────────────────────────────────────────────────────────────────
# ADAPTIVE GRID
# ──────────────────────────────────────────────────────────────────────────────
def build_grid(crs, aoi_ll: dict, native_m: float = 10.0) -> tuple:
    aoi_sc = _aoi_scene(aoi_ll, crs.to_string())
    minx, miny, maxx, maxy = aoi_sc.bounds
    dx = max(maxx - minx, 1e-6)
    dy = max(maxy - miny, 1e-6)
    long_side = max(dx, dy)
    res  = float(np.clip(long_side / MAX_PX_LONG, MIN_RES_M, MAX_RES_M))
    res  = max(res, native_m / 2.0)
    W    = max(1, int(math.ceil(dx / res)))
    H    = max(1, int(math.ceil(dy / res)))
    tf   = Affine.translation(minx, maxy) * Affine.scale(res, -res)
    return aoi_sc, tf, H, W, res


# ──────────────────────────────────────────────────────────────────────────────
# BAND READ HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def _read_band(src, geom_sc, H: int, W: int,
               dst_tf, resamp=Resampling.bilinear) -> np.ndarray:
    win = from_bounds(*geom_sc.bounds, src.transform).round_offsets().round_lengths()
    if win.width <= 0 or win.height <= 0:
        return np.full((H, W), np.nan, "float32")
    arr    = src.read(1, window=win, masked=True).filled(0).astype("float32")
    src_tf = win_transform(win, src.transform)
    dst    = np.full((H, W), np.nan, "float32")
    reproject(arr, dst,
              src_transform=src_tf, src_crs=src.crs,
              dst_transform=dst_tf, dst_crs=src.crs,
              src_nodata=0.0, dst_nodata=np.nan, resampling=resamp)
    return dst


def _read_scl(src, geom_sc, H: int, W: int, dst_tf) -> np.ndarray:
    win = from_bounds(*geom_sc.bounds, src.transform).round_offsets().round_lengths()
    win = win.intersection(Window(0, 0, src.width, src.height)).round_offsets().round_lengths()
    if win.width <= 0 or win.height <= 0:
        return np.zeros((H, W), "int16")
    arr    = src.read(1, window=win, masked=True).filled(0).astype("int16")
    src_tf = win_transform(win, src.transform)
    dst    = np.zeros((H, W), "int16")
    reproject(arr, dst,
              src_transform=src_tf, src_crs=src.crs,
              dst_transform=dst_tf, dst_crs=src.crs,
              src_nodata=0, dst_nodata=0, resampling=Resampling.nearest)
    return dst


# ──────────────────────────────────────────────────────────────────────────────
# FETCH ALL SOIL BANDS  (B02, B04, B08, B11, B12, SCL)
# ──────────────────────────────────────────────────────────────────────────────
_BAND_KEYS = {
    "B02": ("blue",   "B02"),
    "B04": ("red",    "B04"),
    "B08": ("nir",    "B08"),
    "B11": ("swir16", "B11"),
    "B12": ("swir22", "B12"),
}


def fetch_all_bands(item, aoi: dict, dst_tf, H: int, W: int) -> dict | None:
    assets = item.assets
    urls   = {}
    for band, keys in _BAND_KEYS.items():
        url = _pick_url(assets, *keys)
        if not url:
            print(f"  [fetch] missing {band}")
            return None
        urls[band] = url

    scl_ref = assets.get("scl") or assets.get("SCL")
    scl_url = _prefer_https(scl_ref) if scl_ref else None

    try:
        with rasterio.open(urls["B04"]) as ref:
            crs     = ref.crs
            geom_sc = _aoi_scene(aoi, crs.to_string())

        result = {}
        for band, url in urls.items():
            with rasterio.open(url) as ds:
                arr = _read_band(ds, geom_sc, H, W, dst_tf)
            finite = arr[np.isfinite(arr)]
            if finite.size > 0 and finite.max() > 2.0:
                arr /= 10000.0          # DN → reflectance
            result[band] = arr

        if scl_url:
            with rasterio.open(scl_url) as ds:
                result["SCL"] = _read_scl(ds, geom_sc, H, W, dst_tf)
        else:
            result["SCL"] = None

        return result

    except Exception as exc:
        print(f"  [fetch] error: {exc}")
        return None


# ──────────────────────────────────────────────────────────────────────────────
# COMPUTE SPATIAL SPECTRAL INDEX MAPS  (pixel-wise 2-D arrays)
# ──────────────────────────────────────────────────────────────────────────────
def _safe_div(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(np.abs(b) > 1e-9, a / b, np.nan).astype("float32")


def _smooth(arr: np.ndarray, sigma: float) -> np.ndarray:
    """Mask-aware Gaussian smoothing — no halo bleeding from NaN pixels."""
    valid = np.isfinite(arr)
    if not np.any(valid):
        return arr
    num = gaussian_filter(np.where(valid, arr, 0.0).astype("float32"), sigma)
    wgt = gaussian_filter(valid.astype("float32"), sigma)
    return np.where(wgt > 1e-6, num / wgt, np.nan).astype("float32")


def compute_index_maps(bands: dict) -> dict:
    """
    Returns dict of spatial float32 arrays:
      NDVI, BSI, SAVI, SWIR1, CLAY, NDWI
    SCL cloud/shadow mask applied first.
    """
    B02  = bands["B02"];  B04  = bands["B04"]
    B08  = bands["B08"];  B11  = bands["B11"];  B12 = bands["B12"]
    SCL  = bands.get("SCL")

    # ── indices ──────────────────────────────────────────────────────────────
    ndvi  = _safe_div(B08 - B04, B08 + B04)                        # [-1, 1]
    ndwi  = _safe_div(B08 - B12, B08 + B12)                        # moisture
    L     = 0.5
    savi  = ((B08 - B04) / np.where((B08 + B04 + L) != 0,
                                     B08 + B04 + L, np.nan)) * (1 + L)
    savi  = savi.astype("float32")
    bsi_n = (B11 + B04) - (B08 + B02)
    bsi_d = (B11 + B04) + (B08 + B02)
    bsi   = _safe_div(bsi_n, bsi_d)                                # bare soil
    swir1 = B11.copy()                                              # moisture proxy
    clay  = _safe_div(B11, B12)                                     # clay minerals

    # ── SCL masking (clouds + shadows) ───────────────────────────────────────
    if SCL is not None:
        bad = np.isin(SCL, [8, 9, 10, 11] + ([3] if USE_SHADOW else []))
        for arr in [ndvi, ndwi, savi, bsi, swir1, clay]:
            arr[bad] = np.nan

    # ── per-index smoothing ───────────────────────────────────────────────────
    ndvi  = _smooth(ndvi,  SMOOTH_SIGMA)
    ndwi  = _smooth(ndwi,  SMOOTH_SIGMA)
    savi  = _smooth(savi,  SMOOTH_SIGMA)
    bsi   = _smooth(bsi,   SMOOTH_SIGMA)
    swir1 = _smooth(swir1, SMOOTH_SIGMA)
    clay  = _smooth(clay,  SMOOTH_SIGMA)

    return {"NDVI": ndvi, "NDWI": ndwi, "SAVI": savi,
            "BSI":  bsi,  "SWIR1": swir1, "CLAY": clay}


# ──────────────────────────────────────────────────────────────────────────────
# PIXEL-WISE SOIL PROPERTY MAPS
# Literature-calibrated — same formulas as soil_analysis_satellite.py but
# applied pixel-by-pixel to produce SPATIAL maps for VRA.
# ──────────────────────────────────────────────────────────────────────────────
def _clamp_arr(a: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(a, lo, hi).astype("float32")


def compute_soil_maps(idx: dict) -> dict:
    """
    Returns spatial float32 maps (same shape as index maps):
      SOC_map  [%]       Soil Organic Carbon
      N_map    [kg/ha]   Available Nitrogen proxy
      P_map    [kg/ha]   Available Phosphorus proxy
      K_map    [kg/ha]   Available Potassium proxy
    """
    ndvi  = idx["NDVI"]
    savi  = idx["SAVI"]
    bsi   = idx["BSI"]
    swir1 = idx["SWIR1"]
    clay  = idx["CLAY"]

    # SOC:  higher vegetation cover + lower bare-soil exposure → more organic C
    # Ref: Dalal & Henry 1986; Guo et al. 2019
    soc = _clamp_arr(0.8 + 1.5 * ndvi - 0.6 * bsi, 0.1, 5.0)

    # N:  strongly tied to SOC mineralisation (soil N ≈ SOC × 14 × 10 kg/ha)
    n_map = _clamp_arr(soc * 140.0, 50.0, 600.0)

    # P:  vegetation response to phosphorus captured via SAVI
    # Ref: Piekarczyk et al. 2016
    p_map = _clamp_arr(15.0 + 40.0 * (savi + 0.2), 10.0, 80.0)

    # K:  clay mineralogy is the dominant K sink; B11/B12 ratio is clay proxy
    # Ref: Drury 1987; Farifteh et al. 2007
    k_map = _clamp_arr(80.0 + 120.0 * (clay - 0.7), 50.0, 350.0)

    return {"SOC": soc, "N": n_map, "P": p_map, "K": k_map}


# ──────────────────────────────────────────────────────────────────────────────
# ZONE CLASSIFICATION  (per-pixel Low / Medium / High)
# ──────────────────────────────────────────────────────────────────────────────
SOC_THRESHOLDS = (0.5, 1.5)    # % :  Low < 0.5  ≤ Medium ≤ 1.5 < High
N_THRESHOLDS   = (100, 280)    # kg/ha
P_THRESHOLDS   = (15,  40)     # kg/ha
K_THRESHOLDS   = (100, 220)    # kg/ha

_THRESH = {
    "SOC": SOC_THRESHOLDS,
    "N":   N_THRESHOLDS,
    "P":   P_THRESHOLDS,
    "K":   K_THRESHOLDS,
}


def classify_map(arr: np.ndarray, nutrient: str) -> np.ndarray:
    """
    Returns int8 array:  0=nodata  1=Low  2=Medium  3=High
    Note: Low soil content = zone needs HIGH fertiliser dose.
    """
    lo, hi = _THRESH[nutrient]
    out = np.zeros_like(arr, dtype="int8")
    valid = np.isfinite(arr)
    out[valid & (arr <  lo)] = 1   # Low
    out[valid & (arr >= lo) & (arr <= hi)] = 2   # Medium
    out[valid & (arr >  hi)] = 3   # High
    return out


# ──────────────────────────────────────────────────────────────────────────────
# VRA APPLICATION RATE CALCULATION  (per zone, per nutrient)
# ──────────────────────────────────────────────────────────────────────────────
def calc_vra_rates(zone_map: np.ndarray, nutrient: str,
                   crop: str, res_m: float) -> dict:
    """
    For a single nutrient's zone map, compute:
      - area of each zone (ha and acres)
      - recommended product dose per zone (kg/ha)
      - actual product to apply (kg total)
      - text label

    Returns dict with keys 'Low', 'Medium', 'High'.
    """
    M2_HA   = 10000.0
    HA_ACRE = 2.47105

    demand = CROP_DEMAND.get(crop, CROP_DEMAND["default"])
    crop_n_demand = demand[nutrient]                    # kg/tonne yield
    max_dose      = FERTILISER_MAX_DOSE[nutrient]       # kg/ha
    prod          = FERTILISER_PRODUCTS[nutrient]

    px_area_m2 = res_m * res_m
    result     = {}

    for zone_label, zone_int in ZONE_INT.items():
        cnt_px = int(np.count_nonzero(zone_map == zone_int))
        area_m2   = cnt_px * px_area_m2
        area_ha   = area_m2 / M2_HA
        area_ac   = area_ha * HA_ACRE

        frac     = ZONE_DOSE_FRACTION[zone_label]
        dose_nut = round(max_dose * frac, 1)           # kg nutrient / ha
        # convert to product kg/ha
        dose_prod = round(dose_nut / (prod["nutrient_pct"] / 100.0), 1)
        total_kg  = round(dose_prod * area_ha, 1)

        result[zone_label] = {
            "pixel_count": cnt_px,
            "area_ha":     round(area_ha,  3),
            "area_acres":  round(area_ac,  3),
            "nutrient_dose_kg_ha":  dose_nut,
            "product":     prod["name"],
            "product_dose_kg_ha":   dose_prod,
            "total_product_kg":     total_kg,
        }

    return result


# ──────────────────────────────────────────────────────────────────────────────
# IMAGE RENDERING HELPERS
# ──────────────────────────────────────────────────────────────────────────────
_ZONE_CMAP = mcolors.ListedColormap(
    ["#000000",                # 0 = nodata → transparent later
     ZONE_COLORS["Low"],       # 1 = Low soil  (high dose needed)
     ZONE_COLORS["Medium"],    # 2 = Medium
     ZONE_COLORS["High"]])     # 3 = High soil (low dose needed)
_ZONE_NORM = BoundaryNorm([0, 1, 2, 3, 4], _ZONE_CMAP.N)


def _arr_to_b64(fig: plt.Figure) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=DPI,
                bbox_inches="tight", pad_inches=0.05, transparent=True)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _add_zone_legend(ax, title: str = "Soil Level"):
    patches = [
        mpatches.Patch(color=ZONE_COLORS["Low"],    label="Low  (High dose needed)"),
        mpatches.Patch(color=ZONE_COLORS["Medium"], label="Medium (Mod. dose)"),
        mpatches.Patch(color=ZONE_COLORS["High"],   label="High  (Low dose needed)"),
    ]
    ax.legend(handles=patches, loc="lower left", fontsize=6,
              framealpha=0.85, title=title, title_fontsize=7)


def render_soc_map(soc_arr: np.ndarray, res_m: float,
                   capture_date: str) -> str:
    """
    High-quality SOC spatial map.
    Returns base64 PNG string.
    """
    M2_HA   = 10000.0
    HA_ACRE = 2.47105
    px_area = res_m * res_m

    valid = np.isfinite(soc_arr)
    if not np.any(valid):
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center")
        ax.axis("off")
        return _arr_to_b64(fig)

    # tight crop
    rows = np.where(valid.any(axis=1))[0]
    cols = np.where(valid.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    soc_crop = soc_arr[r0:r1, c0:c1]
    valid_crop = np.isfinite(soc_crop)

    # classify into 5 bins for a richer display
    soc_bins = np.full_like(soc_crop, np.nan)
    soc_bins[valid_crop & (soc_crop < 0.3)]                         = 0.15  # Very Low
    soc_bins[valid_crop & (soc_crop >= 0.3) & (soc_crop < 0.7)]    = 0.5   # Low
    soc_bins[valid_crop & (soc_crop >= 0.7) & (soc_crop < 1.2)]    = 0.95  # Medium
    soc_bins[valid_crop & (soc_crop >= 1.2) & (soc_crop < 2.0)]    = 1.6   # High
    soc_bins[valid_crop & (soc_crop >= 2.0)]                        = 2.5   # Very High

    # area stats
    stats = {}
    labels_bins = [
        ("Very Low  (<0.3%)",  0.15, "#3d1c00"),
        ("Low  (0.3–0.7%)",   0.5,  "#c17f00"),
        ("Medium (0.7–1.2%)", 0.95, "#e8c840"),
        ("High  (1.2–2.0%)",  1.6,  "#4caf50"),
        ("Very High (>2.0%)", 2.5,  "#1b5e20"),
    ]
    for lbl, val, _ in labels_bins:
        cnt = int(np.count_nonzero(valid_crop & (soc_bins == val)))
        area_ha = cnt * px_area / M2_HA
        stats[lbl] = {"ha": round(area_ha, 3),
                      "acres": round(area_ha * HA_ACRE, 3)}

    fig = plt.figure(figsize=(10, 7), facecolor="#1a1a2e")
    fig.suptitle(f"SOIL ORGANIC CARBON (SOC) MAP\nCapture: {capture_date}  |  10-m Sentinel-2",
                 color="white", fontsize=13, fontweight="bold", y=0.98)

    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1], wspace=0.05)
    ax_map  = fig.add_subplot(gs[0])
    ax_info = fig.add_subplot(gs[1])

    # ── SOC map panel ────────────────────────────────────────────────────────
    vmin_soc = 0.1; vmax_soc = 3.0
    img = np.ma.masked_invalid(soc_crop)
    im  = ax_map.imshow(img, cmap=CMAP_SOC,
                        vmin=vmin_soc, vmax=vmax_soc,
                        interpolation="bilinear")
    ax_map.set_axis_off()
    ax_map.set_title("SOC Spatial Distribution", color="white",
                     fontsize=10, pad=6)

    # colorbar
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("bottom", size="4%", pad=0.08)
    cb  = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label("SOC (%)", color="white", fontsize=9)
    cb.ax.xaxis.set_tick_params(color="white")
    plt.setp(cb.ax.xaxis.get_ticklabels(), color="white", fontsize=8)
    cax.set_facecolor("#1a1a2e")

    # ── Info panel ───────────────────────────────────────────────────────────
    ax_info.set_facecolor("#1a1a2e")
    ax_info.set_axis_off()

    # legend swatches
    y = 0.95
    ax_info.text(0.05, y, "SOC CLASSES", color="white",
                 fontsize=9, fontweight="bold", va="top",
                 transform=ax_info.transAxes)
    y -= 0.07
    for lbl, val, color in labels_bins:
        area_ha  = stats[lbl]["ha"]
        area_ac  = stats[lbl]["acres"]
        pct_area = area_ha / max(sum(s["ha"] for s in stats.values()), 1e-9) * 100
        rect = mpatches.FancyBboxPatch(
            (0.04, y - 0.025), 0.08, 0.04,
            boxstyle="round,pad=0.005",
            linewidth=0, facecolor=color,
            transform=ax_info.transAxes, clip_on=False)
        ax_info.add_patch(rect)
        ax_info.text(0.16, y - 0.005,
                     f"{lbl}\n{area_ha:.2f} ha  ({pct_area:.0f}%)",
                     color="white", fontsize=6.5, va="top",
                     transform=ax_info.transAxes)
        y -= 0.11

    # summary stats
    y -= 0.05
    mean_soc = float(np.nanmean(soc_crop))
    min_soc  = float(np.nanmin(soc_crop))
    max_soc  = float(np.nanmax(soc_crop))
    total_ha = sum(s["ha"] for s in stats.values())

    for line in [
        "─" * 22,
        f"Mean SOC : {mean_soc:.2f} %",
        f"Min  SOC : {min_soc:.2f} %",
        f"Max  SOC : {max_soc:.2f} %",
        f"Field area: {total_ha:.2f} ha",
        "─" * 22,
        "Low < 0.5%  → Add compost",
        "0.5-1.5%   → Moderate OM",
        "> 1.5%     → Good SOC",
    ]:
        ax_info.text(0.05, y, line, color="#cccccc", fontsize=6.5,
                     va="top", transform=ax_info.transAxes,
                     fontfamily="monospace")
        y -= 0.06

    return _arr_to_b64(fig)


def render_zone_map_single(zone_arr: np.ndarray,
                           nutrient: str,
                           vra_rates: dict,
                           res_m: float,
                           capture_date: str) -> str:
    """Single-nutrient VRA zone map.  Returns base64 PNG."""
    valid = np.isfinite(zone_arr.astype(float)) & (zone_arr > 0)
    if not np.any(valid):
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center")
        ax.axis("off")
        return _arr_to_b64(fig)

    rows = np.where(valid.any(axis=1))[0]
    cols = np.where(valid.any(axis=0))[0]
    crop = zone_arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    img  = np.ma.masked_equal(crop, 0)

    prod = FERTILISER_PRODUCTS[nutrient]["name"]

    fig = plt.figure(figsize=(8, 6), facecolor="#1a1a2e")
    fig.suptitle(
        f"VRA ZONE MAP — {nutrient}  |  {prod}\n"
        f"Capture: {capture_date}",
        color="white", fontsize=11, fontweight="bold", y=0.99)

    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1], wspace=0.04)
    ax_map  = fig.add_subplot(gs[0])
    ax_info = fig.add_subplot(gs[1])

    ax_map.imshow(img, cmap=_ZONE_CMAP, norm=_ZONE_NORM,
                  interpolation="nearest")
    ax_map.set_axis_off()
    _add_zone_legend(ax_map, title=f"Soil {nutrient} Level")

    ax_info.set_facecolor("#1a1a2e")
    ax_info.set_axis_off()

    y = 0.97
    ax_info.text(0.05, y, f"APPLICATION RATES\n{prod}",
                 color="white", fontsize=8, fontweight="bold", va="top",
                 transform=ax_info.transAxes)
    y -= 0.14

    zone_order = [("Low", "HIGH dose"), ("Medium", "MED dose"), ("High", "LOW dose")]
    for zone_lbl, dose_label in zone_order:
        d = vra_rates.get(zone_lbl, {})
        if d.get("pixel_count", 0) == 0:
            continue
        color = ZONE_COLORS[zone_lbl]
        rect = mpatches.FancyBboxPatch(
            (0.03, y - 0.04), 0.10, 0.065,
            boxstyle="round,pad=0.005",
            linewidth=0, facecolor=color,
            transform=ax_info.transAxes, clip_on=False)
        ax_info.add_patch(rect)
        lines = [
            f"Zone: {zone_lbl} soil",
            f"({dose_label})",
            f"{d['product_dose_kg_ha']} kg/ha",
            f"Area: {d['area_ha']:.2f} ha",
            f"Total: {d['total_product_kg']:.0f} kg",
        ]
        for i, line in enumerate(lines):
            ax_info.text(0.18, y - 0.005 - i * 0.028, line,
                         color="white", fontsize=6, va="top",
                         transform=ax_info.transAxes)
        y -= 0.24

    return _arr_to_b64(fig)


def render_combined_4panel(soil_maps: dict,
                            zone_maps: dict,
                            soc_arr: np.ndarray,
                            res_m: float,
                            capture_date: str) -> str:
    """
    4-panel figure: SOC | N zones | P zones | K zones.
    Returns base64 PNG.
    """
    def _crop(arr):
        valid = np.isfinite(arr) & (arr > 0 if arr.dtype.kind == 'i' else True)
        if not np.any(np.isfinite(arr)):
            return arr
        v2 = np.isfinite(arr)
        if not np.any(v2):
            return arr
        rows = np.where(v2.any(axis=1))[0]
        cols = np.where(v2.any(axis=0))[0]
        if rows.size == 0 or cols.size == 0:
            return arr
        return arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor="#0f0f23")
    fig.suptitle(
        f"VARIABLE RATE APPLICATION  —  FIELD MANAGEMENT ZONES\n"
        f"Sentinel-2 L2A  |  Scene: {capture_date}",
        color="white", fontsize=13, fontweight="bold")
    fig.subplots_adjust(hspace=0.15, wspace=0.1)

    panels = [
        ("SOC",   "Soil Organic Carbon (%)",     soc_arr,          None,      CMAP_SOC, (0.1, 3.0)),
        ("N",     "Nitrogen Zones (VRA)",         zone_maps["N"],   "N",       None, None),
        ("P",     "Phosphorus Zones (VRA)",       zone_maps["P"],   "P",       None, None),
        ("K",     "Potassium Zones (VRA)",        zone_maps["K"],   "K",       None, None),
    ]

    for ax, (key, title, arr, nut, cmap, vrange) in zip(axes.flat, panels):
        ax.set_facecolor("#0f0f23")
        cropped = _crop(arr)
        if key == "SOC":
            img = np.ma.masked_invalid(cropped)
            im  = ax.imshow(img, cmap=cmap,
                            vmin=vrange[0], vmax=vrange[1],
                            interpolation="bilinear")
            cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
            cb.set_label("SOC %", color="white", fontsize=8)
            cb.ax.yaxis.set_tick_params(color="white")
            plt.setp(cb.ax.yaxis.get_ticklabels(), color="white", fontsize=7)
        else:
            img = np.ma.masked_equal(cropped, 0)
            ax.imshow(img, cmap=_ZONE_CMAP, norm=_ZONE_NORM,
                      interpolation="nearest")
            _add_zone_legend(ax, title=f"Soil {nut}")

        ax.set_title(title, color="white", fontsize=9, pad=4)
        ax.set_axis_off()

    return _arr_to_b64(fig)


# ──────────────────────────────────────────────────────────────────────────────
# TEXT REPORT
# ──────────────────────────────────────────────────────────────────────────────
def build_text_report(vra_result: dict) -> str:
    meta    = vra_result["metadata"]
    soc_stats = vra_result["soc_stats"]
    vra     = vra_result["vra_rates"]
    crop    = vra_result["crop"]

    hr = "═" * 70
    lines = [
        hr,
        "  VARIABLE RATE APPLICATION (VRA) FIELD REPORT",
        f"  Crop        : {crop}",
        f"  Scene date  : {meta['capture_date']}",
        f"  Cloud cover : {meta['cloud_cover']} %",
        f"  Data source : {meta['collection']}",
        f"  Grid        : {meta['grid_H']} × {meta['grid_W']} px  @  {meta['res_m']:.1f} m/px",
        hr,
        "",
        "  ── SOIL ORGANIC CARBON (SOC) SUMMARY ─────────────────────────",
        f"  Mean  : {soc_stats['mean_pct']:.2f} %",
        f"  Min   : {soc_stats['min_pct']:.2f} %",
        f"  Max   : {soc_stats['max_pct']:.2f} %",
        f"  Field : {soc_stats['total_area_ha']:.2f} ha  ({soc_stats['total_area_acres']:.2f} acres)",
        "",
    ]

    for soc_class, data in soc_stats["classes"].items():
        lines.append(f"  {soc_class:<28}: {data['ha']:.3f} ha  ({data['acres']:.3f} ac)")

    lines += ["", hr, "  ── VRA NUTRIENT ZONES & APPLICATION RATES ────────────────────", ""]

    for nut in ["N", "P", "K"]:
        prod = FERTILISER_PRODUCTS[nut]
        lines.append(f"  ► {nut} — {prod['name']}  ({prod['nutrient_pct']:.0f}% nutrient)")
        lines.append(f"    {'Zone':<10} {'Area (ha)':>10} {'Area (ac)':>10} "
                     f"{'Dose kg/ha':>12} {'Total kg':>10}")
        lines.append(f"    {'─'*10} {'─'*10} {'─'*10} {'─'*12} {'─'*10}")
        zone_data = vra[nut]
        for zone_lbl in ["Low", "Medium", "High"]:
            d = zone_data.get(zone_lbl, {})
            if d.get("pixel_count", 0) == 0:
                continue
            lines.append(
                f"    {zone_lbl:<10} {d['area_ha']:>10.3f} {d['area_acres']:>10.3f} "
                f"{d['product_dose_kg_ha']:>12.1f} {d['total_product_kg']:>10.1f}")
        lines.append("")

    lines += [
        hr,
        "  ── INTERPRETATION ────────────────────────────────────────────",
        "  Zone 'Low'   = Soil is nutrient-deficient  → Apply FULL dose",
        "  Zone 'Medium'= Moderate soil content       → Apply 65% dose",
        "  Zone 'High'  = Soil is nutrient-rich       → Apply only 30% dose",
        "  (VRA saves fertiliser cost on High zones and prevents over-application)",
        hr,
    ]
    return "\n".join(lines)


# ──────────────────────────────────────────────────────────────────────────────
# MAIN VRA PIPELINE
# ──────────────────────────────────────────────────────────────────────────────
def run_vra_analysis(aoi_geojson: dict,
                     start_date:  str,
                     end_date:    str,
                     crop:        str = "wheat") -> dict:
    """
    Full pipeline:
      STAC search → fetch bands (B02 B04 B08 B11 B12 SCL) →
      compute index maps → compute soil property maps →
      classify zones (Low/Med/High) per N, P, K, SOC →
      compute VRA application rates per zone →
      render 3 output images (SOC map, VRA NPK map, combined 4-panel) →
      build text report →
      return everything

    Returns
    -------
    dict with keys:
      soc_map_b64        : base64 PNG string  — SOC spatial map
      vra_npk_b64        : base64 PNG string  — 3 single-nutrient zone maps (N, P, K)
      combined_b64       : base64 PNG string  — 4-panel combined map
      text_report        : str                — full text report
      vra_rates          : dict               — per-nutrient per-zone application data
      soc_stats          : dict               — SOC area statistics
      metadata           : dict               — scene info
      crop               : str
    """
    print(f"\n{'═'*60}")
    print(f"  VRA ZONING PIPELINE")
    print(f"  Crop: {crop}   AOI window: {start_date} → {end_date}")
    print(f"{'═'*60}")

    # ── 1. Scene search ───────────────────────────────────────────────────────
    print("\n[1/6] Searching best Sentinel-2 scene ...")
    item, cap_date, cloud = find_best_scene(aoi_geojson, start_date, end_date)
    if item is None:
        raise RuntimeError("No Sentinel-2 scene found for this AOI / date range.")

    collection = getattr(item, "collection_id", None) or "sentinel-2-l2a"

    # ── 2. Build adaptive grid ────────────────────────────────────────────────
    print("[2/6] Building adaptive target grid ...")
    assets  = item.assets
    red_url = _pick_url(assets, "red", "B04")
    if not red_url:
        raise RuntimeError("No red-band asset found — cannot derive CRS.")
    with rasterio.open(red_url) as ref:
        crs = ref.crs
    aoi_sc, dst_tf, H, W, res_m = build_grid(crs, aoi_geojson, native_m=10.0)
    print(f"       Grid: {H} × {W} px  @  {res_m:.2f} m/px")

    # ── 3. Fetch all bands ────────────────────────────────────────────────────
    print("[3/6] Fetching bands (B02 B04 B08 B11 B12 + SCL) ...")
    bands = fetch_all_bands(item, aoi_geojson, dst_tf, H, W)
    if bands is None:
        raise RuntimeError("Band fetch failed — check network / STAC availability.")

    # Apply AOI mask
    aoi_mask = geometry_mask(
        [mapping(aoi_sc)], out_shape=(H, W),
        transform=dst_tf, invert=True)
    for b in ["B02", "B04", "B08", "B11", "B12"]:
        bands[b][~aoi_mask] = np.nan

    # ── 4. Compute index maps ─────────────────────────────────────────────────
    print("[4/6] Computing spectral index maps (NDVI, BSI, SAVI, SWIR1, CLAY) ...")
    idx_maps  = compute_index_maps(bands)

    # ── 5. Derive soil property maps + classify zones ─────────────────────────
    print("[5/6] Deriving SOC / N / P / K spatial maps & classifying zones ...")
    soil_maps = compute_soil_maps(idx_maps)

    soc_arr  = soil_maps["SOC"]
    n_arr    = soil_maps["N"]
    p_arr    = soil_maps["P"]
    k_arr    = soil_maps["K"]

    zone_maps = {
        "N": classify_map(n_arr,  "N"),
        "P": classify_map(p_arr,  "P"),
        "K": classify_map(k_arr,  "K"),
    }

    # VRA rates per nutrient
    vra_rates = {}
    for nut, z_map in zone_maps.items():
        vra_rates[nut] = calc_vra_rates(z_map, nut, crop, res_m)

    # SOC area statistics
    M2_HA   = 10000.0
    HA_ACRE = 2.47105
    px_area = res_m * res_m
    valid_soc = np.isfinite(soc_arr)
    soc_classes_out = {}
    soc_class_defs = [
        ("Very Low  (<0.3%)",  (None,  0.3)),
        ("Low  (0.3–0.7%)",   (0.3,   0.7)),
        ("Medium (0.7–1.2%)", (0.7,   1.2)),
        ("High  (1.2–2.0%)",  (1.2,   2.0)),
        ("Very High (>2.0%)", (2.0,  None)),
    ]
    total_valid_px = int(np.count_nonzero(valid_soc))
    for lbl, (lo, hi) in soc_class_defs:
        if lo is None:
            mask = valid_soc & (soc_arr < hi)
        elif hi is None:
            mask = valid_soc & (soc_arr >= lo)
        else:
            mask = valid_soc & (soc_arr >= lo) & (soc_arr < hi)
        cnt = int(np.count_nonzero(mask))
        ha  = cnt * px_area / M2_HA
        soc_classes_out[lbl] = {
            "pixels": cnt,
            "ha":     round(ha, 3),
            "acres":  round(ha * HA_ACRE, 3),
            "pct_area": round(cnt / max(total_valid_px, 1) * 100, 1),
        }

    total_ha = total_valid_px * px_area / M2_HA
    soc_stats = {
        "mean_pct":       round(float(np.nanmean(soc_arr)), 3),
        "min_pct":        round(float(np.nanmin(soc_arr)),  3),
        "max_pct":        round(float(np.nanmax(soc_arr)),  3),
        "std_pct":        round(float(np.nanstd(soc_arr)),  3),
        "total_area_ha":  round(total_ha, 3),
        "total_area_acres": round(total_ha * HA_ACRE, 3),
        "classes":        soc_classes_out,
    }

    # ── 6. Render images ──────────────────────────────────────────────────────
    print("[6/6] Rendering output images ...")

    # 6a. SOC map (standalone, high-quality)
    soc_b64 = render_soc_map(soc_arr, res_m, cap_date)
    print("       ✓ SOC map rendered")

    # 6b. Three individual VRA maps (N, P, K) stacked vertically
    vra_figs = []
    for nut in ["N", "P", "K"]:
        b64 = render_zone_map_single(
            zone_maps[nut], nut, vra_rates[nut], res_m, cap_date)
        vra_figs.append(b64)
    print("       ✓ Individual VRA zone maps (N, P, K) rendered")

    # 6c. Combined 4-panel
    combined_b64 = render_combined_4panel(
        soil_maps, zone_maps, soc_arr, res_m, cap_date)
    print("       ✓ Combined 4-panel map rendered")

    # ── Assemble result ───────────────────────────────────────────────────────
    result = {
        # Images (base64 PNG)
        "soc_map_b64":     soc_b64,
        "vra_n_b64":       vra_figs[0],
        "vra_p_b64":       vra_figs[1],
        "vra_k_b64":       vra_figs[2],
        "combined_b64":    combined_b64,
        # Data
        "vra_rates":       vra_rates,
        "soc_stats":       soc_stats,
        "crop":            crop,
        "metadata": {
            "capture_date": cap_date,
            "cloud_cover":  cloud,
            "collection":   collection,
            "grid_H": H,
            "grid_W": W,
            "res_m":  res_m,
        },
        # Mean index values (for debugging / audit)
        "mean_indices": {k: round(float(np.nanmean(v)), 5)
                         for k, v in idx_maps.items()},
    }

    result["text_report"] = build_text_report(result)
    print("\n" + result["text_report"])

    return result


# ──────────────────────────────────────────────────────────────────────────────
# OPTIONAL: save all images to disk
# ──────────────────────────────────────────────────────────────────────────────
def save_images(result: dict, out_dir: str = ".") -> None:
    os.makedirs(out_dir, exist_ok=True)
    pairs = [
        ("soc_map_b64",  "soc_map.png"),
        ("vra_n_b64",    "vra_nitrogen.png"),
        ("vra_p_b64",    "vra_phosphorus.png"),
        ("vra_k_b64",    "vra_potassium.png"),
        ("combined_b64", "vra_combined_4panel.png"),
    ]
    for key, fname in pairs:
        b64 = result.get(key)
        if b64:
            path = os.path.join(out_dir, fname)
            with open(path, "wb") as f:
                f.write(base64.b64decode(b64))
            print(f"  Saved → {path}")


# ──────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Example AOI: Maharashtra field (can be any polygon in EPSG:4326)
    aoi_geojson = {
        "type": "Polygon",
        "coordinates": [[
            [77.15618780277907, 20.137282565715527],
            [77.15840867184340, 20.136496870757790],
            [77.15745380543410, 20.135580221647306],
            [77.15575864933669, 20.136134240983303],
            [77.15618780277907, 20.137282565715527],
        ]]
    }

    START_DATE = "2024-11-01"
    END_DATE   = "2024-12-21"
    CROP       = "onion"         # change to any crop in CROP_DEMAND dict

    result = run_vra_analysis(
        aoi_geojson,
        START_DATE,
        END_DATE,
        crop=CROP,
    )

    # ── Print JSON-safe data summary ──────────────────────────────────────────
    summary = {k: v for k, v in result.items()
               if not k.endswith("_b64")}   # exclude image blobs from print
    print("\n── JSON DATA SUMMARY ────────────────────────────────────────────")
    print(json.dumps(summary, indent=2, default=str))

    # ── Save all PNG images to ./output/ ─────────────────────────────────────
    save_images(result, out_dir="./vra_output")
    print("\nDone. Check ./vra_output/ for PNG maps.")


════════════════════════════════════════════════════════════
  VRA ZONING PIPELINE
  Crop: wheat   AOI window: 2026-0-01 → 2026-05-21
════════════════════════════════════════════════════════════

[1/6] Searching best Sentinel-2 scene ...


ValueError: time data '2026-0-01' does not match format '%Y-%m-%d'